# Afternoon class 26/08 — Worksheet 13 SOLUTIONS: combined challenges   (L04)

Every cell below was executed on the same Python the lab ships; the quoted
output is real.

Each answer is followed by the reasoning, because on this sheet the reasoning
IS the answer — several of these produce a number that is arithmetically
perfect and substantively misleading, and the commentary is where that gets
named.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Worksheet 13 — Combined challenges. Run this once.
#
# One tuple per ORDER LINE. An order with three products is three lines
# here, all sharing an OrderID -- exactly like superstore.orders on 24/08.
#
#              OrderID  Customer  Region     Category           Qty   UnitPrice
order_lines = [
    (1001,    "Ada",    "East",    "Technology",       1,  1200.00),
    (1001,    "Ada",    "East",    "Office Supplies",  3,     4.50),
    (1001,    "Ada",    "East",    "Furniture",        1,   340.00),
    (1002,    "Bo",     "West",    "Technology",       2,   650.00),
    (1003,    "Ada",    "East",    "Office Supplies", 10,     4.50),
    (1004,    "Cai",    "Central", "Furniture",        1,   890.00),
    (1004,    "Cai",    "Central", "Office Supplies",  5,    12.00),
    (1005,    "Dee",    "North",   "Technology",       1,   320.00),
    (1005,    "Dee",    "North",   "Technology",       1,    99.00),
    (1005,    "Dee",    "North",   "Office Supplies",  2,     4.50),
    (1005,    "Dee",    "North",   "Furniture",        2,   150.00),
    (1006,    "Bo",     "West",    "Office Supplies",  1,    22.00),
]

# A second structure, to be joined onto the first later.
customer_city = {"Ada": "Toronto", "Bo": "Vancouver", "Cai": "Montreal"}

print("order lines loaded:", len(order_lines))
print("first line:", order_lines[0])

CHALLENGE 1 — How big is this dataset, really?

Before any analysis, three different numbers could all be called "the size"
of this data, and they disagree. Find all three.

### Question 1a

Three different sizes, all of them correct.

In [ ]:
print("order lines:     ", len(order_lines))

orders = {line[0] for line in order_lines}
print("distinct orders: ", len(orders))

customers = {line[1] for line in order_lines}
print("distinct customers:", len(customers), customers)

-> order lines `12` | distinct orders `6` | distinct customers `4`,
`{'Ada', 'Bo', 'Cai', 'Dee'}` in some order.

All three describe the same data and none of them is wrong. They answer
different questions — and `len(order_lines)` is the one you get for free
without thinking, which is exactly why it is the one that ends up in
reports by accident.

(The customer set prints in an arbitrary order. Worksheet 05 again: never
lean on it.)

### Question 1b

Answer in words.

In [ ]:
# ANSWER: 6. That is the number of distinct OrderIDs.
#
# Reporting 12 (the line count) would inflate order volume by 2x, because an
# order with three products is three rows here. Reporting 4 (customers) would
# answer a different question entirely -- how many people bought, not how
# many times they bought.
#
# len(order_lines) is the tempting one precisely because it needs no thought.

-> **6.**

Note the line count here happens to be exactly twice the order count. That
is a coincidence of this small dataset, not a relationship — so you cannot
even apply a correction factor after the fact. You have to count the right
thing in the first place.

CHALLENGE 2 — Where does the revenue come from?

Each line's revenue is quantity times unit price. Nothing in the data holds
that figure; you have to compute it.

### Question 2a

Revenue by category, biggest first.

In [ ]:
revenue = {}
for order_id, customer, region, category, qty, price in order_lines:
    if category not in revenue:      # first time we have seen this category
        revenue[category] = 0
    revenue[category] = revenue[category] + qty * price

for category, total in sorted(revenue.items(), key=lambda pair: pair[1], reverse=True):
    print(category, round(total, 2))

print("grand total:", round(sum(revenue.values()), 2))

-> `Technology 2919.0` | `Furniture 1530.0` | `Office Supplies 149.5`,
grand total `4598.5`.

THE ACCUMULATOR PATTERN: start with an empty dict; for each row, create the
key with a zero if it is missing, then add to it. You will write this loop
more often than any other in Python, and the `if category not in revenue`
line is the whole of it.

SORTING BY VALUE: `sorted()` on a dict gives you its keys, which is not what
you wanted. `d.items()` gives `(key, value)` pairs, and
`key=lambda pair: pair[1]` tells `sorted` to compare the second element of
each pair — the value — instead of the first.

The totals print as `2919.0`, not `2919.00`. `round()` returns a number, and
a number has no opinion about trailing zeros. Presentation is Challenge 6's
job.

### Question 2b

Answer in words.

In [ ]:
# ANSWER: not established. Technology is the biggest by REVENUE, and that is
# all we measured. It comes from 4 lines with high unit prices, while Office
# Supplies is the category almost every order touches.
#
# Revenue, margin, order frequency and customer reach are four different
# questions. We answered one and the ranking would very likely change under
# the others -- we have no cost data at all, so profitability is unknown.

-> **Not established.**

WHAT WE ACTUALLY MEASURED was revenue, and only revenue. Office Supplies
appears in 5 of the 6 orders while Technology appears in 3, so ranked by
reach the order inverts completely.

Neither ranking is "the" answer. The question "which category matters most"
was never well defined, and the number quietly picked a definition on your
behalf. We also have no cost data at all, so profitability — usually the
thing actually being asked about — is simply unknown.

CHALLENGE 3 — The biggest order

Same accumulator pattern, different key. This is the one place the
line-versus-order distinction does real work.

### Question 3a

Revenue per order, ranked.

In [ ]:
order_total = {}
for order_id, customer, region, category, qty, price in order_lines:
    if order_id not in order_total:
        order_total[order_id] = 0
    order_total[order_id] = order_total[order_id] + qty * price

ranked = sorted(order_total.items(), key=lambda pair: pair[1], reverse=True)
for order_id, total in ranked:
    print(order_id, round(total, 2))

print("biggest order:", ranked[0])

-> `1001 1553.5` | `1002 1300.0` | `1004 950.0` | `1005 728.0` |
`1003 45.0` | `1006 22.0`. Biggest: `(1001, 1553.5)`.

This is Challenge 2's code with one word changed: `category` became
`order_id`. That is the real lesson — once your data is a list of tuples,
"group by X" is always this same loop, and choosing X is the only decision
that matters.

`ranked[0]` works because `sorted` returned a list, so it is indexable. Be
aware that if two orders tied for first this would silently pick whichever
one `sorted` happened to put first, and tell you nothing about the tie.

### Question 3b

Answer in words.

In [ ]:
# ANSWER: no. The single biggest LINE is order 1002's technology line at
# 1300.00, which beats any one line in order 1001 (the largest of those is
# 1200.00). Order 1001 only wins once its three lines are added together.
#
# So "biggest order" and "biggest line" name different rows of the data. If
# you had grouped by the wrong one you would have got a clean, confident,
# wrong answer -- with nothing in the output to hint at it.

-> **No.**

The biggest single LINE is order 1002's technology line at 2 x 650.00 =
`1300.00`. Order 1001's largest line is only `1200.00`; it reaches 1553.50
solely because its three lines are summed.

So "biggest order" and "biggest line" have different winners on this data —
which is lucky, because it makes the mistake visible. On a larger dataset
the two would coincide often enough that you might never notice you had
asked the wrong question.

CHALLENGE 4 — Customer reach

A dictionary whose values are SETS. This is the shape to reach for whenever
the question is "which distinct things does each X touch".

### Question 4a

Distinct categories touched, per customer.

In [ ]:
bought = {}
for order_id, customer, region, category, qty, price in order_lines:
    if customer not in bought:
        bought[customer] = set()      # a set, so repeats collapse for free
    bought[customer].add(category)

for customer, categories in bought.items():
    print(customer, len(categories), sorted(categories))

all_three = {c for c in bought if len(bought[c]) == 3}
print("touched all three:", all_three)

-> `Ada 3` | `Bo 2` | `Cai 2` | `Dee 3`, and `{'Ada', 'Dee'}` touched all
three.

A DICT OF SETS. The value starts as `set()` rather than `0`, and you `add`
to it rather than adding a number. Reach for this shape whenever the
question contains the word "distinct".

Note `sorted(categories)` in the print. The sets have no order of their own,
so sorting at the moment of display gives stable, readable output without
pretending the underlying structure is ordered.

### Question 4b

Answer in words.

In [ ]:
# ANSWER: because the question asks for DISTINCT categories, and a set
# enforces that at the moment of insertion -- no checking, no deduplicating
# step to forget.
#
# With a list, Ada would show 4 categories rather than 3: she bought Office
# Supplies on both order 1001 and order 1003, so it would appear twice. The
# count would be wrong and nothing would look wrong -- you would just be
# counting purchases while believing you were counting categories.

-> **Because the question says DISTINCT, and a set enforces that on
insertion.**

WITH A LIST, Ada would report 4 categories. She bought Office Supplies on
order 1001 and again on order 1003, so it would be stored twice. The count
would be wrong, the output would look entirely normal, and you would be
counting purchases while the variable was named `categories`.

Choosing the container IS the deduplication step — there is no second line
to forget later.

CHALLENGE 5 — Joining two structures

`customer_city` is a second, separate structure. Attaching it to the order
data is the Python equivalent of the joins you wrote on 24/08 — and it has
the same failure mode.

### Question 5a

Revenue per customer, joined to city.

In [ ]:
by_customer = {}
for order_id, customer, region, category, qty, price in order_lines:
    if customer not in by_customer:
        by_customer[customer] = 0
    by_customer[customer] = by_customer[customer] + qty * price

for customer, total in by_customer.items():
    city = customer_city.get(customer, "UNKNOWN")   # [] would raise KeyError
    print(customer, city, round(total, 2))

print("customers with no city:", {c for c in by_customer if c not in customer_city})

-> `Ada Toronto 1598.5` | `Bo Vancouver 1322.0` | `Cai Montreal 950.0` |
`Dee UNKNOWN 728.0`, and `{'Dee'}` has no city.

`.get(customer, "UNKNOWN")` is doing the work of a LEFT JOIN: keep the row
even when the lookup fails, and mark it as unmatched.
`customer_city[customer]` would have raised `KeyError` on Dee and stopped
the report — which, note, is the SAFER of the two failures, because it is
loud.

The last line is the habit worth taking from this challenge: having joined,
immediately count what did not match.

### Question 5b

Answer in words.

In [ ]:
# ANSWER: Dee's 728.00 would disappear -- 15.8% of the 4598.50 total.
#
# And the report would still balance internally, still sum to a plausible
# number, and give no sign anything was missing. This is the single most
# common way a join quietly lies to you: the rows that fail to match are the
# rows you never see.
#
# The habit that catches it: after any join, count what did NOT match and
# report that count, even when you expect it to be zero.

-> **728.00 would vanish — 15.8% of the 4598.50 total.**

AND THE REPORT WOULD LOOK FINE. Three cities, plausible figures, a total
internally consistent with the rows shown. Nothing on the page would be
false. The page would simply be missing a sixth of the business, with
nothing to indicate it.

This is precisely the failure mode of an INNER JOIN used where a LEFT JOIN
was meant, and it is why "how many rows did not match" is worth printing
even when you are confident the answer is zero.

CHALLENGE 6 — A report someone could actually read

Same numbers as Challenge 2, presented properly. `enumerate` supplies the
rank and an f-string does the alignment.

### Question 6a

The same numbers, formatted as a report.

In [ ]:
ranked_categories = sorted(revenue.items(), key=lambda pair: pair[1], reverse=True)

for rank, (category, total) in enumerate(ranked_categories, start=1):
    print(f"{rank}. {category:<16} {total:>10.2f}")

print(f"   {'TOTAL':<16} {sum(revenue.values()):>10.2f}")

->
```
1. Technology          2919.00
2. Furniture           1530.00
3. Office Supplies      149.50
   TOTAL               4598.50
```

The figures now line up and the decimals are consistent: `{total:>10.2f}`
forces two decimal places, so `149.5` displays as `149.50` and `2919.0` as
`2919.00`. Compare Challenge 2's ragged output.

FORMAT ONLY AT THE EDGE. The stored values are still full-precision floats;
nothing in the data was rounded. Round for display, never inside the
calculation, or the rounding errors compound as they accumulate.

### Question 6b

Answer in words.

In [ ]:
# ANSWER: because enumerate hands over a 2-item tuple whose second item is
# ITSELF a 2-item tuple -- (1, ("Technology", 2919.0)). The inner brackets
# unpack that nested tuple in the same statement.
#
# Writing `for rank, category, total in ...` would raise ValueError: Python
# would be given 2 things to unpack into 3 names. The bracket structure on
# the left has to mirror the structure of the data on the right.

-> **Because the second item `enumerate` hands over is itself a tuple.**

Each pass gives `(1, ('Technology', 2919.0))` — two items, the second of
which contains two more. `for rank, (category, total)` mirrors that shape
exactly.

Writing `for rank, category, total in ...` raises
`ValueError: not enough values to unpack (expected 3, got 2)`. The bracket
structure to the left of `in` must match the structure of the data to the
right — the same rule as worksheet 03 Q7, nested one level deeper.

CHALLENGE 7 — Inverting a dictionary

A reasonable-looking transformation with a trap in it. Read the lengths
before you read anything else.

### Question 7a

Orders per category, then inverted.

In [ ]:
orders_per_category = {}
for order_id, customer, region, category, qty, price in order_lines:
    if category not in orders_per_category:
        orders_per_category[category] = set()
    orders_per_category[category].add(order_id)

counts = {category: len(ids) for category, ids in orders_per_category.items()}
print(counts, len(counts))

inverted = {count: category for category, count in counts.items()}
print(inverted, len(inverted))

-> `{'Technology': 3, 'Office Supplies': 5, 'Furniture': 3}` with length
`3`, then `{3: 'Furniture', 5: 'Office Supplies'}` with length **`2`**.

READ THE TWO LENGTHS BEFORE ANYTHING ELSE. Three pairs went in, two came
out, and Python raised nothing at all.

### Question 7b

Answer in words.

In [ ]:
# ANSWER: Technology disappeared. It and Furniture both appear in 3 distinct
# orders, so both wanted the key 3 -- and since Furniture is processed later,
# it overwrote Technology. Last one wins, exactly as in worksheet 07 Q9.
#
# Nothing failed. `inverted` is a perfectly valid dictionary containing a
# quietly wrong answer.
#
# Inverting is safe only when the VALUES are known to be unique, and "known"
# has to mean checked. The check is one line: compare len() before and after,
# and treat any shrinkage as data loss rather than a curiosity.

-> **Technology disappeared, because it tied with Furniture on 3.**

Both wanted the key `3`. Furniture is processed later, so it overwrote —
"last one wins", the duplicate-key rule from worksheet 07 Q9, turning up
somewhere you were not watching for it. `inverted` is a perfectly valid
dictionary containing a quietly wrong answer.

WHEN INVERTING IS SAFE: only when the values are known to be unique, and
"known" has to mean checked. Counts, statuses, categories, dates — these
are exactly the fields that repeat, so inverting a dictionary keyed on
anything like them is unsafe by default.

THE CHECK IS ONE LINE: compare `len()` before and after, and treat any
shrinkage as data loss rather than a curiosity.

CHALLENGE 8 — Stretch: the executive summary

One cell, the headline numbers, the way they would appear on a slide. Every
one of them has a grain trap somewhere near it.

### Question 8a

Stretch — the executive summary.

In [ ]:
total_revenue = sum([line[4] * line[5] for line in order_lines])
n_lines = len(order_lines)
n_orders = len({line[0] for line in order_lines})
n_customers = len({line[1] for line in order_lines})
best_category, best_revenue = max(revenue.items(), key=lambda pair: pair[1])

print("total revenue:      ", round(total_revenue, 2))
print("order lines:        ", n_lines)
print("orders:             ", n_orders)
print("customers:          ", n_customers)
print("average ORDER value:", round(total_revenue / n_orders, 2))
print("average LINE value: ", round(total_revenue / n_lines, 2))
print("best category:      ", best_category, round(best_revenue, 2))

# cross-check: the per-order totals must sum to the same grand total
print("cross-check:", round(sum(order_total.values()), 2) == round(total_revenue, 2))

-> total revenue `4598.5` | order lines `12` | orders `6` | customers `4` |
average ORDER value `766.42` | average LINE value `383.21` | best category
`Technology 2919.0` | cross-check `True`.

The cross-check is the most valuable line in the cell. Summing the per-order
totals from Challenge 3 must reproduce a grand total computed a completely
different way — if it did not, one of the two loops has a bug. It costs one
line, and it is the only thing here capable of catching a mistake.

### Question 8b

Answer in words — and the point of the whole sheet.

In [ ]:
# ANSWER: 766.42 is the average ORDER value (total / 6 orders). 383.21 is the
# average LINE value (total / 12 lines). Both are arithmetically correct.
#
# "Average order value" means the first one. The second answers "average
# value of a product line within an order", which is a real number that
# almost nobody wants and that nobody would question on a slide, because it
# is labelled plausibly and sits next to figures that are correct.
#
# The tell is the denominator, and the denominator comes from the GRAIN of
# the data. On 24/08 the same trap was COUNT(*) versus COUNT(DISTINCT
# OrderID) on superstore.orders. It is the same mistake in a different
# language: len(order_lines) is not the number of orders, and it never was.

-> **766.42 is per ORDER. 383.21 is per LINE. Both are arithmetically
perfect.**

"Average order value" means the first. The second answers "average value of
a product line within an order" — a real quantity that essentially nobody
wants, that sits happily on a slide beside correct figures, and that no-one
in the room will challenge because it is labelled plausibly.

THE TELL IS ALWAYS THE DENOMINATOR, and the denominator comes from the
GRAIN of the data — what one row actually represents. Here one row is a
line, not an order.

YOU HAVE MET THIS BEFORE. On 24/08 it was `COUNT(*)` versus
`COUNT(DISTINCT OrderID)` on `superstore.orders`, and it produced most of
the wrong answers in that session. Different language, identical mistake:
`len(order_lines)` is not the number of orders, and it never was.

Writing the loop is the easy half. Knowing which denominator the question
deserves is the half that makes the number mean anything.